# Python Containerization (Docker Deep Dive)

In [ ]:
# Day 40 — Python Containerization
#
# You probably know basic Docker: FROM, COPY, RUN, CMD.
# But containerizing Python for production (especially ML models) is Tricky.
#
# Common Pitfalls:
# 1. Massive Image Sizes: 'python:3.11' is ~1GB. Add PyTorch and you're at 4GB.
# 2. Rebuilding Dependencies: If you COPY everything before pip install, Docker rebuilds
#    all your packages every time you change 1 line of code.
# 3. Security: Running as 'root' inside a container is a security risk.
# 4. WSGI/ASGI Servers: Running `python app.py` for FastAPI in production will fail
#    under load. You need `gunicorn` with `uvicorn` workers.
#
# Today we'll cover:
#   1. Slim Images vs Alpine
#   2. Layer Caching (The correct order of COPY and RUN)
#   3. Multi-stage Builds for ML
#   4. Running Securely as Non-Root
#   5. The Production Entrypoint

### 1. The Naive Dockerfile (What NOT to do)

In [ ]:
import os
os.makedirs('docker_demo', exist_ok=True)

# This is how most beginners write a Python Dockerfile:
naive_dockerfile = '''
FROM python:3.11

# Setting working directory
WORKDIR /app

# Copying everything FIRST
COPY . .

# Installing dependencies
RUN pip install -r requirements.txt

# Running the app
CMD ["python", "app.py"]
'''

with open('docker_demo/Dockerfile.naive', 'w') as f:
    f.write(naive_dockerfile)

print("Why is this bad?")
print("1. Base Image: 'python:3.11' contains compilers, headers, and OS tools we don't need (Very Large).")
print("2. Caching: Because 'COPY . .' happens BEFORE 'RUN pip install', ANY change to ANY file")
print("   will bust the Docker cache. Pip will reinstall all ML libraries (taking minutes) on every build.")
print("3. Security: It runs as root.")

### 2. The Optimized Dockerfile

In [ ]:
# Here is the optimized way to build a Python Container

good_dockerfile = '''
# 1. Use the -slim version. 
# (Note: Don't use -alpine for Python. Alpine uses 'musl' instead of 'glibc', 
# which breaks many pre-compiled C-extensions like pandas/numpy, forcing them to compile from source!)
FROM python:3.11-slim

# Prevent Python from writing .pyc files and buffering stdout
ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

WORKDIR /app

# 2. OPTIMIZE CACHING: Copy ONLY the requirements first
COPY requirements.txt .

# Now run pip install. 
# Because requirements.txt rarely changes, Docker caches this layer!
# --no-cache-dir keeps the image small by removing pip's download cache.
RUN pip install --no-cache-dir -r requirements.txt

# 3. NOW copy the rest of your code. 
# Code changes frequently, but this layer is fast to rebuild.
COPY . .

# 4. SECURITY: Create a non-root user and switch to it
RUN useradd -m appuser && chown -R appuser /app
USER appuser

# 5. Run the ASGI server (for FastAPI)
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('docker_demo/Dockerfile.optimized', 'w') as f:
    f.write(good_dockerfile)

print("Optimized Dockerfile created! Review the inline comments to understand the fixes.")

### 3. Multi-Stage Builds (For ML Dependencies)

In [ ]:
# Sometimes you need 'gcc' or 'g++' to compile a Python package (like xgboost from source).
# But you DON'T want compilers in your final production image (bloat + security risk).
# We solve this with Multi-Stage builds.

multi_stage = '''
# STAGE 1: Builder
FROM python:3.11-slim AS builder
WORKDIR /app

# Install compilers (needed to build wheels)
RUN apt-get update && apt-get install -y gcc build-essential

COPY requirements.txt .
# Install packages into a specific directory (/install)
RUN pip install --prefix=/install -r requirements.txt

# --------------------------------------------
# STAGE 2: Runtime (Final Image)
FROM python:3.11-slim
WORKDIR /app

# Copy only the COMPILED python packages from the builder stage
COPY --from=builder /install /usr/local

# Copy application code
COPY . .

USER 1000
CMD ["python", "predict.py"]
'''

with open('docker_demo/Dockerfile.multistage', 'w') as f:
    f.write(multi_stage)

print("Multi-stage Dockerfile created!")
print("Stage 1 installs compilers and builds the packages.")
print("Stage 2 copies ONLY the built packages, leaving the compilers behind.")

### 4. Running Production Web Servers

In [ ]:
# `uvicorn app:app` runs a SINGLE process.
# If your ML prediction takes 1 second, and 5 users hit the API at once,
# the 5th user waits 5 seconds.
#
# In production, we use Gunicorn as a process manager to run MULTIPLE Uvicorn workers.
# 
# Install: pip install gunicorn uvicorn
# Command:
# gunicorn app:app --workers 4 --worker-class uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000

print("For production FastAPI Deployments:")
print("Always use Gunicorn to manage multiple Uvicorn workers to utilize all CPU cores!")

In [ ]:
# Practice Questions:
#
# 1. Why is alpine Linux generally a bad choice for Python Data Science containers?
#
# 2. Why should `COPY . .` come AFTER `RUN pip install` in a Dockerfile?
#
# 3. What is the benefit of a multi-stage Docker build?

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The Ultimate ML Dockerfile
# Combine what you've learned to write the ultimate Dockerfile for an ML model:
# - Use a slim base image.
# - Set environment variables to prevent bytecode.
# - Optimize the cache layer for requirements.
# - Run as a non-root user.
# - Use gunicorn + uvicorn for the entrypoint.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Cost: Slim images save cloud storage and transfer costs.
# 2. Speed: Optimized caching means CI/CD pipelines finish in seconds instead of minutes.
# 3. Reliability: Gunicorn ensures your API doesn't hang when multiple users request predictions.
# 4. Security: Non-root execution is a mandatory compliance requirement in enterprise Kubernetes.